# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We'll use the Croissant schema for the FAIR² Mental Health Survey Dataset, which contains multiple record sets, fields, and columns relevant to mental health and demographic factors.

### Dataset Source
The dataset is defined by a Croissant schema available at:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access the metadata object and print key details
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print("Keywords:", metadata.keywords)

## 2. Data Overview
Review available record sets, fields, and their IDs. All references to entities such as record sets and fields should use their `@id` values.

In [ ]:
# List all record sets defined in the dataset (by @id)
record_sets_info = dataset.record_sets()
print("Record Sets:")
for rs in record_sets_info:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')} | description: {rs.get('description', 'N/A')}")

# For illustration, list fields (columns) in each record set:
for rs in record_sets_info:
    print(f"\nFields for Record Set @id={rs['@id']}:")
    fields_info = dataset.fields(record_set=rs['@id'])
    for f in fields_info:
        print(f"  - Field @id: {f['@id']} | name: {f.get('name','N/A')} | dataType: {f.get('dataType','N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id values for loading
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

# Load records from each record set into a pandas DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"Record Set @id: {record_set_id} has no records.")

# For further steps, pick the first populated record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
else:
    first_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, including filtering records based on specific criteria, normalizing numeric fields, and grouping data. Always reference fields by their `@id` values.

In [ ]:
# Example EDA using one record set and numeric field
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Assume the first record set has a GAD-7 field (replace with actual @id as needed)
if first_record_set_id:
    df = dataframes[first_record_set_id]

    # Identify numeric fields (using `dataType` from Croissant fields schema)
    fields_info = dataset.fields(record_set=first_record_set_id)
    numeric_fields = [f['@id'] for f in fields_info if f.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']]
    print("Numeric fields:", numeric_fields)

    # For demo, use the first numeric field
    if numeric_fields:
        numeric_field_id = numeric_fields[0]

        # Filter records with value above threshold
        threshold = 10
        if numeric_field_id in df.columns:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())

            # Normalize the numeric field
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by a demographic field (e.g., gender)
            demo_fields = [f['@id'] for f in fields_info if f.get('dataType') == 'schema:Text']
            if demo_fields:
                group_field_id = demo_fields[0]
                if group_field_id in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
                    print(grouped_df.head())
        else:
            print(f"Field {numeric_field_id} not found in DataFrame columns.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No record sets with loaded data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization for the numeric field (if present)
import matplotlib.pyplot as plt

if first_record_set_id and numeric_fields and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df was created above, show a bar plot
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id], color='salmon')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, and analyze a dataset described by a Croissant schema using the `mlcroissant` library.

- We loaded metadata and surveyed the structure using `@id` references for record sets and fields.
- Data extraction respects schema-decoupled entity references using Croissant's `@id`.
- Key numeric fields can be filtered, normalized, and grouped by demographic attributes.
- Visualizations can reveal key trends or distributions, supporting public health and AI research in mental health contexts.

To build deeper analyses, extend this notebook with additional filtering or modeling steps using the field and record set `@id`s as shown.